# Experimentos — Instâncias Small

Objetivo: rodar o ciclo completo do PBIL-Fuzzy (`src/engine/pbil_fuzzy.py`,
Fase 6) sobre as instâncias em `data/instances/Small/`, inspecionar a
convergência de uma execução em detalhe, e comparar o resultado final
entre todas as instâncias pequenas.

**Aviso:** com a FAM ainda placeholder (Fase 5, `TODO(aluno)`), os
números aqui servem para validar que o pipeline roda ponta a ponta —
não são resultados finais de qualidade do algoritmo. Rode de novo
depois de calibrar o fuzzy.

Depende de `src/core/`, `src/pbil/`, `src/fuzzy/`, `src/engine/`
(Fases 3 a 6) e de `config.yaml` na raiz do projeto.

In [ ]:
import sys
import os

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))


In [ ]:
import time

import numpy as np
import pandas as pd
import yaml

from src.core.instance import carregar_instancia, listar_instancias
from src.engine.pbil_fuzzy import PBILFuzzy
from src.utils import visualization


## Carregar configuração e listar instâncias Small


In [ ]:
caminho_config = os.path.join("..", "config.yaml")
with open(caminho_config, "r", encoding="utf-8") as arquivo:
    config = yaml.safe_load(arquivo)

diretorio_small = os.path.join("..", config["instancia"]["diretorio_small"])
caminhos_instancias_small = listar_instancias(diretorio_small)

print(f"Instâncias encontradas em {diretorio_small}: {len(caminhos_instancias_small)}")
for caminho in caminhos_instancias_small:
    print(" -", os.path.basename(caminho))


## Execução detalhada — primeira instância

Roda o algoritmo completo numa única instância e inspeciona a
convergência do `Cmax_best`, a evolução de `alpha`/`beta` e da
diversidade estrutural ao longo das gerações.

In [ ]:
assert len(caminhos_instancias_small) > 0, (
    "Nenhuma instância encontrada em data/instances/Small — "
    "coloque pelo menos um arquivo .txt lá antes de rodar esta célula."
)

instancia_detalhe = carregar_instancia(caminhos_instancias_small[0])
print(instancia_detalhe)

motor_detalhe = PBILFuzzy(instancia_detalhe, config)

tempo_inicio = time.time()
resultado_detalhe = motor_detalhe.executar()
tempo_execucao = time.time() - tempo_inicio

print(f"\nCmax_best final: {resultado_detalhe.cmax_best:.2f}")
print(f"Gerações executadas: {len(resultado_detalhe.historico_cmax_best)}")
print(f"Tempo de execução: {tempo_execucao:.2f}s")


In [ ]:
visualization.plotar_convergencia_cmax(
    resultado_detalhe.historico_cmax_best,
    titulo=f"Convergência do Cmax — {instancia_detalhe.nome}",
)


In [ ]:
visualization.plotar_evolucao_alpha_beta(
    resultado_detalhe.historico_alpha,
    resultado_detalhe.historico_beta,
    titulo=f"Evolução de α e β — {instancia_detalhe.nome}",
)


In [ ]:
visualization.plotar_diversidade_estrutural(
    resultado_detalhe.historico_diversidade_estrutural,
    titulo=f"Diversidade estrutural — {instancia_detalhe.nome}",
)


## Comparação entre todas as instâncias Small

Roda uma execução única (não múltiplas repetições — isso fica pro
`scripts/run_experiments.py` da Fase 7) em cada instância de
`data/instances/Small/`, e monta uma tabela comparativa de
`Cmax_best` final.

In [ ]:
linhas_resumo = []

for caminho in caminhos_instancias_small:
    instancia = carregar_instancia(caminho)
    motor = PBILFuzzy(instancia, config, gerador_aleatorio=np.random.default_rng(config["execucao"]["seed"]))

    tempo_inicio = time.time()
    resultado = motor.executar()
    tempo_execucao = time.time() - tempo_inicio

    linhas_resumo.append({
        "instancia": instancia.nome,
        "numero_jobs": instancia.numero_jobs,
        "numero_maquinas": instancia.numero_maquinas,
        "cmax_best": resultado.cmax_best,
        "geracoes": len(resultado.historico_cmax_best),
        "tempo_segundos": round(tempo_execucao, 2),
    })

    print(f"{instancia.nome}: Cmax_best={resultado.cmax_best:.2f} ({tempo_execucao:.2f}s)")

tabela_resumo = pd.DataFrame(linhas_resumo)
tabela_resumo


## Conclusão

O ciclo completo do PBIL-Fuzzy roda ponta a ponta sobre as instâncias
Small, produzindo histórico de convergência, evolução de
`alpha`/`beta` e diversidade estrutural. Para experimentos com
múltiplas execuções independentes por instância (necessário para
qualquer conclusão estatística), usar `scripts/run_experiments.py`
(Fase 7).

Lembrete: recalibrar o fuzzy (Fase 5, Seção 8) antes de gerar
resultados para o relatório final.